# Stage 3: Joint Mediation Analysis

In this final stage, we synthesize all data into a master annual dataset and perform formal Mediation Testing to evaluate the causal chain:
**Climate Shock (Rainfall) -> Food Price Volatility -> External Debt**

We use the **Baron & Kenny** approach for statistical mediation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import os

sns.set(style="whitegrid")

## 3.1 Master Dataset Construction
Aggregating all variables to an annual national level.

In [ ]:
# Load Datasets
rf_df = pd.read_csv('datasets/bgd-rainfall-subnat-full.csv')
food_df = pd.read_csv('datasets/wfp_food_prices_bgd.csv')
debt_df = pd.read_csv('datasets/external-debt_bgd.csv')

rf_df['year'] = pd.to_datetime(rf_df['date']).dt.year
food_df['year'] = pd.to_datetime(food_df['date']).dt.year

# Annual Aggregations
rf_annual = rf_df.groupby('year')['rfq'].agg(['mean', 'std']).reset_index()
rf_annual.columns = ['year', 'annual_rfq', 'rf_variability']

food_annual = food_df.groupby('year')['price'].agg(['mean', 'std']).reset_index()
food_annual.columns = ['year', 'avg_food_price', 'food_volatility']

debt_filtered = debt_df[debt_df['Indicator Name'] == 'External debt stocks, total (DOD, current US$)'].copy()
debt_annual = debt_filtered[['Year', 'Value']].rename(columns={'Year': 'year', 'Value': 'total_debt'})
debt_annual['debt_growth'] = debt_annual['total_debt'].pct_change()

# Merge to Master
master_df = pd.merge(rf_annual, food_annual, on='year')
master_df = pd.merge(master_df, debt_annual, on='year').dropna()

print(f"Master Dataset Shape: {master_df.shape}")
display(master_df.head())

## 3.2 Formal Mediation Test (Baron & Kenny)
Evaluating the three paths of mediation.

In [ ]:
X = sm.add_constant(master_df['annual_rfq'])
M = master_df['food_volatility']
Y = master_df['debt_growth']

# Path 1: X -> Y (Direct Effect)
model1 = sm.OLS(Y, X).fit()
print("Path 1 (Rain -> Debt Growth) P-val:", model1.pvalues['annual_rfq'])

# Path 2: X -> M (Mediator effect)
model2 = sm.OLS(M, X).fit()
print("Path 2 (Rain -> Food Volatility) P-val:", model2.pvalues['annual_rfq'])

# Path 3: X + M -> Y (Mediation effect)
X_combined = sm.add_constant(pd.concat([master_df['annual_rfq'], M], axis=1))
model3 = sm.OLS(Y, X_combined).fit()
print("Path 3 (Food Volatility -> Debt Growth | Rain) P-val:", model3.pvalues['food_volatility'])

print("\n--- CONCLUSION ---")
if model1.pvalues['annual_rfq'] < 0.05 and model3.pvalues['food_volatility'] < 0.05:
    print("Mediation is statistically supported.")
else:
    print("Mediation is not strongly supported by current annual data sample.")

## 3.3 Full Chain Feature Importance
Using Random Forest to rank which factors most impact Debt Growth.

In [ ]:
features = ['annual_rfq', 'rf_variability', 'avg_food_price', 'food_volatility']
X_rf = master_df[features]
y_rf = master_df['debt_growth']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_rf, y_rf)

plt.figure(figsize=(10, 6))
pd.Series(rf_model.feature_importances_, index=features).sort_values().plot(kind='barh')
plt.title('Stage 3: Feature Importance for Debt Growth prediction')
plt.xlabel('Gini Importance')
plt.show()

## 3.4 Year-by-Year Risk Profiling
Visualizing the timeline of climate-food-debt risk.

In [ ]:
scaler = StandardScaler()
cluster_features = ['annual_rfq', 'food_volatility', 'debt_growth']
X_scaled = scaler.fit_transform(master_df[cluster_features])

master_df['risk_cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(X_scaled)

plt.figure(figsize=(14, 6))
sns.lineplot(data=master_df, x='year', y='debt_growth', marker='o', label='Debt Growth Rate')
plt.fill_between(master_df['year'], master_df['debt_growth'], alpha=0.2, color='gray')
plt.scatter(master_df['year'], master_df['debt_growth'], c=master_df['risk_cluster'], cmap='coolwarm', s=100, zorder=5)
plt.title('Timeline of Debt Growth and Multi-Pillar Risk Clusters')
plt.ylabel('Debt Growth Rate')
plt.legend()
plt.show()